# Lab 03 guiado · Limpieza y análisis exploratorio

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 3, ruta guiada*

Este cuaderno se abre directo desde GitHub. Lo que escribas se pierde al cerrar la pestaña. Si quieres conservarlo, descárgalo con Archivo, Descargar, Descargar .ipynb.

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#31708f"></td><td bgcolor="#d9edf7"><font color="#0b3d62" size="3"><b>Qué vamos a responder</b><br>1. Qué hay que mirar en un archivo antes de calcularle nada.<br>2. Qué hacer con las filas repetidas, los datos que faltan y los valores imposibles.<br>3. Qué tan bien quedó la limpieza, comparada con el dato original.</font></td></tr></table>

---
<br>

## Paso 1 de 9 · ¿Cómo llegó el archivo?

**Qué vamos a responder.** Si el archivo de demanda trae las filas que debería traer.

**Qué hacer.** Corre la celda. Lee el archivo tal como llegó, sin arreglar nada.

In [ ]:
# Paso 1
DATOS = ("https://raw.githubusercontent.com/RodrigoMSB/"
         "curso-python-minergia/main/datos/")
import pandas as pd

# El archivo viene separado por punto y coma, como lo guarda Excel en Chile.
sucia = pd.read_csv(DATOS + "demanda_sucia.csv", sep=";")

# 2024 fue bisiesto. 366 días por 24 horas por 7 regiones.
esperadas = 366 * 24 * 7
print("Filas del archivo ", len(sucia))
print("Filas esperadas   ", esperadas)
print("Sobran            ", len(sucia) - esperadas)
sucia.head()

---
<br>

## Paso 2 de 9 · ¿Qué tiene de raro?

**Qué vamos a responder.** Qué problemas trae el archivo antes de calcular nada.

**Qué hacer.** Corre la celda y lee las cuatro líneas que aparecen.

In [ ]:
# Paso 2
print("Regiones distintas     ", sucia["region"].nunique(), " y deberían ser 7")
print("Horas sin dato de mwh  ", sucia["mwh"].isna().sum())

# repr muestra el valor con comillas cuando es texto.
print("Primer valor de mwh    ", repr(sucia["mwh"].iloc[0]), " es texto, no número")
print("Primera fecha          ", repr(sucia["fecha"].iloc[0]), " es texto, no fecha")

---
<br>

## Paso 3 de 9 · Tu turno, mira cómo está escrita cada región

**Qué vamos a responder.** De cuántas formas distintas aparece escrita una misma región.

**Qué hacer.** Mueve el deslizador para ver más filas y vuelve a correr la celda. Las siete primeras son las bien escritas.

In [ ]:
# Paso 3
cuantas = 12 #@param {type:"slider", min:5, max:49, step:1}

# value_counts cuenta cuántas veces aparece cada forma, de mayor a menor.
sucia["region"].value_counts().head(cuantas)

---
<br>

## Paso 4 de 9 · ¿Cómo se ordena el desorden?

**Qué vamos a responder.** Cómo se convierten el número, la fecha y la región a su forma correcta.

![Número, fecha y región convertidos desde texto](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo03/img/lab03_paso4_tipos.png)

**Qué hacer.** Corre la celda. Trabaja sobre una copia, así el archivo original queda intacto.

In [ ]:
# Paso 4
limpia = sucia.copy()

# La coma decimal pasa a punto y el texto pasa a número.
limpia["mwh"] = limpia["mwh"].str.replace(",", ".").astype(float)

# El formato dice cómo viene escrita la fecha, día, mes y año.
limpia["fecha"] = pd.to_datetime(limpia["fecha"], format="%d/%m/%Y")

# strip quita los espacios de los extremos y title deja la primera letra en mayúscula.
limpia["region"] = limpia["region"].str.strip().str.title()

print("✓ Regiones distintas", limpia["region"].nunique())
print("✓ Demanda total del año", round(limpia["mwh"].sum()), "MWh")
print("✓ Fechas del", limpia["fecha"].min().date(), "al", limpia["fecha"].max().date())

---
<br>

## Paso 5 de 9 · ¿Hay filas repetidas?

**Qué vamos a responder.** Si las filas que sobraban en el Paso 1 son copias.

![Una fila repetida se quita](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo03/img/lab03_paso5_repetidas.png)

**Qué hacer.** Corre la celda.

In [ ]:
# Paso 5
# duplicated marca cada fila idéntica a una que ya apareció antes.
print("Filas repetidas", limpia.duplicated().sum())

sin_repetidas = limpia.drop_duplicates()
print("Filas que quedan", len(sin_repetidas))
if len(sin_repetidas) == esperadas:
    print("✓ Ahora calza con las", esperadas, "filas esperadas")

---
<br>

## Paso 6 de 9 · Tu turno, decide qué hacer con los datos que faltan

**Qué vamos a responder.** Cuánto cambia el resultado según lo que se haga con las horas vacías.

![Cuatro formas de rellenar las horas vacías](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo03/img/lab03_paso6_huecos.png)

**Qué hacer.** Elige una estrategia en la lista y vuelve a correr la celda. Prueba las cuatro y compara el promedio y el día de Valparaíso.

In [ ]:
# Paso 6
estrategia = "trazar una recta" #@param ["borrar la fila", "poner cero", "repetir el valor anterior", "trazar una recta"]

# Ordenadas por región y por hora, para que cada hora quede junto a sus vecinas.
orden = sin_repetidas.sort_values(["region", "fecha", "hora"]).reset_index(drop=True)
por_region = orden.groupby("region")["mwh"]

if estrategia == "borrar la fila":
    rellena = orden["mwh"].dropna()
elif estrategia == "poner cero":
    rellena = orden["mwh"].fillna(0)
elif estrategia == "repetir el valor anterior":
    rellena = por_region.ffill()
else:
    rellena = por_region.transform(lambda s: s.interpolate())

print(f"Promedio por hora con {estrategia}", round(rellena.mean(), 1), "MWh")

# Un día de Valparaíso que venía con cuatro horas vacías.
dia = orden[(orden["region"] == "Valparaíso") & (orden["fecha"] == "2024-09-16")]
vista = pd.DataFrame({"hora": dia["hora"],
                      "venía": dia["mwh"],
                      "queda": rellena.reindex(dia.index)})
vista.head(12).round(1).fillna({"venía": "vacía", "queda": "se borra"}).set_index("hora")

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>La decisión que no se ve en el informe</b><br>Las cuatro estrategias dan resultados distintos con los mismos datos.<br>Poner cero baja el promedio y repetir el valor anterior aplana el día.<br>Ninguna es la correcta siempre, pero la que se use se deja escrita.<br>Desde el próximo paso seguimos con trazar una recta.</font></td></tr></table>

---
<br>

## Paso 7 de 9 · Tu turno, decide qué valor es imposible

**Qué vamos a responder.** Qué horas traen un valor que no puede ser real y cómo se corrigen.

![Picos y negativos separados por una regla de negocio](https://raw.githubusercontent.com/RodrigoMSB/curso-python-minergia/main/modulos/modulo03/img/lab03_paso7_imposibles.png)

**Qué hacer.** Corre la celda. Después mueve el deslizador a 1 y a 8, vuelve a correrla y mira cuántas horas marca. En el Paso 9 vas a ver qué tan bien elegiste.

In [ ]:
# Paso 7
veces = 3 #@param {type:"slider", min:1, max:10, step:1}

tabla = sin_repetidas.sort_values(["region", "fecha", "hora"]).reset_index(drop=True)
promedio_region = tabla.groupby("region")["mwh"].transform("mean")

# Dos reglas de negocio. La demanda no puede ser negativa,
# y un valor muchas veces mayor que el promedio de su región es sospechoso.
negativo = tabla["mwh"] < 0
pico = tabla["mwh"] > veces * promedio_region
print("Horas con demanda negativa", negativo.sum())
print(f"Horas con más de {veces} veces el promedio de su región", pico.sum())

# Antes de tocar nada se marca qué horas se van a cambiar y por qué.
final = tabla.copy()
final["fue_nulo"] = final["mwh"].isna()
final["fue_corregido"] = negativo | pico

# Los valores imposibles se vacían y todos los huecos se rellenan con una recta.
final.loc[final["fue_corregido"], "mwh"] = None
final["mwh"] = final.groupby("region")["mwh"].transform(lambda s: s.interpolate())

print("✓ Horas rellenadas porque venían vacías", final["fue_nulo"].sum())
print("✓ Horas corregidas porque eran imposibles", final["fue_corregido"].sum())
print("✓ Horas sin dato que quedan", final["mwh"].isna().sum())

---
<br>

## Paso 8 de 9 · ¿Cuánto consume una región en una hora normal?

**Qué vamos a responder.** Si el promedio describe bien una hora típica.

**Qué hacer.** Corre la celda. Compara el promedio y la mediana de todo el país con los de cada región.

In [ ]:
# Paso 8
# La mediana es el valor del medio. La mitad de las horas está por debajo y la otra mitad por encima.
print("Todo el país junto")
print("  Promedio", round(final["mwh"].mean(), 1), " Mediana", round(final["mwh"].median(), 1))

resumen = final.groupby("region")["mwh"].agg(promedio="mean", mediana="median")
resumen.round(1).sort_values("promedio", ascending=False)

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>El promedio de todo el país no describe a ninguna región</b><br>Con todas las regiones juntas el promedio es el doble de la mediana.<br>No es por valores extremos, que ya se corrigieron.<br>Es porque la Metropolitana consume varias veces más que las demás.<br>Dentro de cada región el promedio y la mediana están casi iguales.</font></td></tr></table>

---
<br>

## Paso 9 de 9 · ¿Qué tan bien quedó la limpieza?

**Qué vamos a responder.** Cuánto se equivocó la limpieza, comparada con el dato original.

**Qué hacer.** Corre la celda. Si cambias el deslizador del Paso 7, corre ese paso y después este otra vez.

In [ ]:
# Paso 9
# Este lab tiene algo que el trabajo real no tiene, el archivo original sin errores.
verdad = pd.read_csv(DATOS + "demanda.csv", parse_dates=["fecha"])

auditoria = pd.merge(final, verdad, on=["fecha", "hora", "region"],
                     suffixes=("_limpia", "_real"))
auditoria["error"] = (auditoria["mwh_limpia"] - auditoria["mwh_real"]).abs()

auditoria["grupo"] = "no se tocó"
auditoria.loc[auditoria["fue_nulo"], "grupo"] = "se rellenó"
auditoria.loc[auditoria["fue_corregido"], "grupo"] = "se corrigió"

print("Horas comparadas", len(auditoria))
auditoria.groupby("grupo")["error"].agg(horas="count", error_promedio="mean",
                                        error_maximo="max").round(1)

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#e0b000"></td><td bgcolor="#fff8d6"><font color="#5c4700" size="3"><b>Esto no se puede hacer en el trabajo real</b><br>Aquí se pudo medir el error porque existía el archivo original.<br>En el trabajo real el dato sucio es el único que hay.<br>Lo que sí se puede hacer siempre es marcar qué horas se rellenaron<br>y cuáles se corrigieron, como hacen fue_nulo y fue_corregido.</font></td></tr></table>

---
<br>

## Lo que lograste hoy

<table width="100%" cellpadding="12" cellspacing="0"><tr><td width="6" bgcolor="#28a745"></td><td bgcolor="#d8f0df"><font color="#14532d" size="3">• Antes de calcular se cuentan las filas y se mira cómo vienen escritos los datos.<br>• Un número con coma o una fecha chilena llegan como texto y hay que convertirlos.<br>• Las filas repetidas se detectan y se quitan antes de sumar.<br>• Cada forma de rellenar un hueco cambia el resultado, por eso se deja escrita.<br>• Un valor imposible se encuentra con una regla de negocio.<br>• El promedio de todo junto puede no describir a ningún grupo.</font></td></tr></table>

## Para seguir practicando

Estos ejercicios son para quien quiera seguir después de clase. Aquí sí se escribe código.
Antes de empezar, corre todo el cuaderno con Entorno de ejecución, Ejecutar todo.
Escribe tu respuesta en la celda vacía de cada ejercicio y abre la solución solo después de intentarlo.

### Ejercicio 1 · Arregla el archivo en la misma lectura

En el Paso 4 el número se arregló después de leer. Vuelve a leer demanda_sucia.csv pidiéndole a read_csv que entienda la coma decimal, y convierte la fecha. Revisa con dtypes que mwh quede como número y fecha como fecha. Pista, read_csv acepta decimal=",".

In [ ]:
# Ejercicio 1

<details>
<summary>Solución · Ejercicio 1</summary>

```python
otra = pd.read_csv(DATOS + "demanda_sucia.csv", sep=";", decimal=",")
otra["fecha"] = pd.to_datetime(otra["fecha"], format="%d/%m/%Y")
otra.dtypes
```

mwh queda como float64 y fecha como datetime64.

</details>

### Ejercicio 2 · ¿Es mejor rellenar con el promedio de la región y la hora?

Suena razonable rellenar una hora vacía con el promedio de esa misma región a esa misma hora del día, porque respeta la forma del día. Hazlo y compara su error con el de trazar una recta, solo en las horas que venían vacías. Pista, agrupa por region y hora y usa transform("mean").

In [ ]:
# Ejercicio 2

<details>
<summary>Solución · Ejercicio 2</summary>

```python
copia = tabla.copy()
copia["fue_nulo"] = copia["mwh"].isna()
copia.loc[negativo | pico, "mwh"] = None

promedio_rh = copia.groupby(["region", "hora"])["mwh"].transform("mean")
copia["mwh"] = copia["mwh"].fillna(promedio_rh)

otra_auditoria = pd.merge(copia, verdad, on=["fecha", "hora", "region"],
                          suffixes=("_limpia", "_real"))
otra_auditoria["error"] = (otra_auditoria["mwh_limpia"] - otra_auditoria["mwh_real"]).abs()

print("Trazando una recta          ",
      round(auditoria.loc[auditoria["fue_nulo"], "error"].mean(), 1), "MWh")
print("Con el promedio región y hora",
      round(otra_auditoria.loc[otra_auditoria["fue_nulo"], "error"].mean(), 1), "MWh")
```

La recta se equivoca 5,4 MWh en promedio y el promedio de la región y la hora 27,0. Lo que suena más razonable no siempre es lo mejor, y por eso se mide.

</details>

### Ejercicio 3 · La demanda alta de cada región

El máximo siempre es un caso raro. Para hablar de demanda alta se usa el percentil 95, el valor que solo supera una de cada veinte horas. Calcúlalo para cada región con la tabla final. Pista, quantile(0.95) después de agrupar.

In [ ]:
# Ejercicio 3

<details>
<summary>Solución · Ejercicio 3</summary>

```python
final.groupby("region")["mwh"].quantile(0.95).round(1).sort_values(ascending=False)
```

La Metropolitana encabeza con 1.189,4 MWh por hora y Atacama cierra con 47,0.

</details>

### Ejercicio 4 · La demanda por franja del día

Agrega a una copia de final una columna franja con cuatro tramos de horas, madrugada de 0 a 5, mañana de 6 a 11, tarde de 12 a 17 y noche de 18 a 23. Después arma una tabla con las regiones en las filas, las franjas en las columnas y la demanda promedio en cada casilla. Pista, pd.cut con bins=[-1, 5, 11, 17, 23] y pivot_table.

In [ ]:
# Ejercicio 4

<details>
<summary>Solución · Ejercicio 4</summary>

```python
con_franja = final.copy()
con_franja["franja"] = pd.cut(con_franja["hora"], bins=[-1, 5, 11, 17, 23],
                              labels=["madrugada", "mañana", "tarde", "noche"])
con_franja.pivot_table(index="region", columns="franja", values="mwh",
                       aggfunc="mean", observed=True).round(1)
```

En todas las regiones la madrugada es la franja más baja y la tarde la más alta.

</details>

Los métodos estadísticos para encontrar valores extremos y el detalle de cada estrategia de relleno están en [la ruta completa de este módulo](https://colab.research.google.com/github/RodrigoMSB/curso-python-minergia/blob/main/modulos/modulo03/lab.ipynb).